# PPE v5 후보 — 하단각도 자동 라벨링 + v4 보강 학습 + 깃허브 업로드 묶음 (Google Colab 한 번에)

**런타임 → 런타임 유형 변경 → T4 GPU** 를 고른 뒤 **런타임 → 모두 실행**. 약 1.5시간.

| 단계 | 셀 | 하는 일 |
| :--- | :--- | :--- |
| A 라벨링 | 1~6 | Grounding DINO(Apache-2.0)로 400장에 `helmet`·`vest` 박스 → 애매한 사진은 자동 제외 · 검수 시트 저장 |
| B 학습 | 7~14 | v4 `best_ckpt.pth` 에서 이어 학습 (기존 `ppe4_cs_md_v2` + 하단각도) · v4 와 비교 · ONNX 변환 |
| C 묶음 | 15 | 팀장 깃허브에 올릴 형식으로 정리 (`github_upload/`) + 업로드 권장/보류 판정 |

## 준비물 — Google Drive `MyDrive/mechadog_ppe_261002/`
| 파일 | |
| :--- | :--- |
| `PPE_하단각도_학습이미지_400.zip` | 받은 zip 그대로 |
| `ppe-v4_handover.zip` | 팀장 전달 묶음 그대로 (`best_ckpt.pth` + `ppe4_cs_md_v2/` + `SHA256SUMS`) |

## 결과 — `MyDrive/mechadog_ppe_261002/out_v5/`
- `github_upload/release_ppe-v5/` — Release 자산 (`ppe.onnx` · `ppe.json` · `NOTICE_addendum.txt`)
- `github_upload/REPO_CHANGES.md` — 저장소에서 고칠 값(해시·크기·태그)과 커밋할 파일 + 업로드 판정
- `label_review/` — 자동 라벨 검수 시트 (학습 뒤에라도 꼭 본다)

⚠️ 왜 v4 로 라벨을 찍지 않나 — v4 가 못 보는 하단각도를 v4 의 답으로 가르치면 보강이 되지 않는다.
⚠️ 새 사진만으로 학습하지 않는다 — 한 사람·한 방·착용 상태로 기울어 미착용 검출이 무너진다.
⚠️ 데이터셋·체크포인트·사진은 깃에 올리지 않는다 (공개 데이터의 사람 사진 · 얼굴 포함).

In [ ]:
# 1. 경로 · Drive 연결
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")


DRIVE = Path("/content/drive/MyDrive/mechadog_ppe_261002")
SRC_ZIP = DRIVE / "PPE_하단각도_학습이미지_400.zip"
HANDOVER_ZIP = DRIVE / "ppe-v4_handover.zip"
OUT = DRIVE / "out_v5"
REVIEW = OUT / "label_review"
REVIEW.mkdir(parents=True, exist_ok=True)
for p in (SRC_ZIP, HANDOVER_ZIP):
    assert p.exists(), f"없음: {p}"

WORK = Path("/content/work")
DATA = WORK / "ppe_mix"  # 학습 데이터 (기존 + 하단각도)
HANDOVER = WORK / "old_raw" / "ppe-v4_handover"
CKPT_V4 = HANDOVER / "best_ckpt.pth"
IMG_DIR = WORK / "low_raw" / "train"  # 하단각도 원본 + 자동 라벨
low_json = IMG_DIR / "_annotations.coco.json"
print("준비물 확인 완료")

In [ ]:
# 2. 설치 — YOLOX (Apache-2.0) · ultralytics 는 쓰지 않는다 (AGPL · ADR-24)
import sys


def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


sh("git clone -q https://github.com/Megvii-BaseDetection/YOLOX /content/YOLOX || true")
sh("cd /content/YOLOX && git checkout -q 0.3.0")
sh("pip -q install loguru tabulate thop ninja pycocotools onnx onnxruntime-gpu")
# ⚠️ `pip install -e .` 은 하지 않는다 — Colab Python 3.13 에서 YOLOX 0.3.0 빌드가 실패한다.
# C++ 확장은 쓰지 않으므로(COCOeval 은 표준판으로 바꾼다) 소스 폴더를 import 경로에 넣기만 한다.
sys.path.insert(0, "/content/YOLOX")
os.environ["PYTHONPATH"] = "/content/YOLOX:" + os.environ.get("PYTHONPATH", "")  # tools/train.py 용
# torch 2.6+ 는 torch.load 기본이 weights_only=True 라 YOLOX 체크포인트를 못 읽는다.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sh(
    "wget -q -nc -P /content https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.onnx"
)
from yolox.exp import Exp  # noqa: E402, F401 — 여기서 실패하면 학습 셀까지 가지 않는다

print("YOLOX import OK")

In [ ]:
# 3. 데이터 풀기 — 팀장 묶음(해시 확인) + 하단각도 사진
import zipfile

shutil.rmtree(WORK, ignore_errors=True)
IMG_DIR.mkdir(parents=True)
sh(f"unzip -q '{HANDOVER_ZIP}' -d {WORK}/old_raw")
sh(f"cd '{HANDOVER}' && sha256sum -c --quiet SHA256SUMS")  # 잘린 업로드·다른 판이면 여기서 멈춘다
shutil.copytree(HANDOVER / "ppe4_cs_md_v2", DATA)
ANN = DATA / "annotations"  # YOLOX COCODataset 은 data_dir/annotations/ 에서 읽는다
assert (ANN / "instances_train.json").exists()

# zip 안 한글 폴더 이름이 깨져도 파일 이름(PPE_lower_0001.jpg)만 쓴다
with zipfile.ZipFile(SRC_ZIP) as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/").rsplit("/", 1)[-1]
        if name.startswith("PPE_lower_") and name.lower().endswith(".jpg"):
            (IMG_DIR / name).write_bytes(z.read(info))
FILES = sorted(IMG_DIR.glob("PPE_lower_*.jpg"))
assert len(FILES) == 400, f"400장이 아니다: {len(FILES)}"
print("기존 데이터:", DATA, "| 하단각도:", len(FILES), "장")

In [ ]:
# 4. [A] 라벨러 불러오기 — Grounding DINO
import torch
from PIL import Image
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

MODEL_ID = "IDEA-Research/grounding-dino-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(device).eval()
PROMPT = "a person. a hard hat. a safety vest."
print("device:", device)


def find(image):
    """[(종류, 점수, (x1, y1, x2, y2))] — 종류는 person / helmet / vest."""
    inputs = processor(images=image, text=PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs)
    kw = {"text_threshold": 0.25, "target_sizes": [image.size[::-1]]}
    try:  # transformers 판에 따라 인자 이름이 다르다
        res = processor.post_process_grounded_object_detection(
            out, inputs.input_ids, threshold=0.25, **kw
        )[0]
    except TypeError:
        res = processor.post_process_grounded_object_detection(
            out, inputs.input_ids, box_threshold=0.25, **kw
        )[0]
    texts = res.get("text_labels", res.get("labels"))
    found = []
    for t, s, b in zip(texts, res["scores"].tolist(), res["boxes"].tolist(), strict=False):
        t = str(t)
        kind = (
            "helmet"
            if "hat" in t
            else "vest"
            if "vest" in t
            else "person"
            if "person" in t
            else None
        )
        if kind:
            found.append((kind, s, tuple(b)))
    return found

In [ ]:
# 5. [A] 자동 라벨링 + 규칙 검사
import csv

SCORE_MIN = {"helmet": 0.35, "vest": 0.35}  # 이 미만이면 박스를 치지 않는다
REVIEW_BELOW = 0.5  # 쳤어도 이 미만이면 검수 목록에 올린다
CLIP_PX = 2  # 안전모 박스 윗변이 화면 위 이 픽셀 이내면 잘린 것으로 본다


def area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])


def inter(a, b):
    return (max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3]))


def ioa(b, r):
    return area(inter(b, r)) / area(b) if area(b) else 0.0


labels, notes = {}, {}
for i, path in enumerate(FILES):
    image = Image.open(path).convert("RGB")
    W, H = image.size
    found = find(image)
    people = [f for f in found if f[0] == "person"]
    person = max(people, key=lambda f: area(f[2]))[2] if people else (0, 0, W, H)
    ph = person[3] - person[1]
    head = (
        person[0],
        person[1] - 0.15 * ph,
        person[2],
        person[1] + 0.35 * ph,
    )  # 챙이 사람 박스 위로 나와도 된다
    picked, why = [], []
    for kind in ("helmet", "vest"):
        cands = [
            f
            for f in found
            if f[0] == kind
            and f[1] >= SCORE_MIN[kind]
            and ioa(f[2], head if kind == "helmet" else person) >= 0.5
        ]
        if not cands:
            why.append(f"{kind} 없음")
            continue
        _, s, b = max(cands, key=lambda f: f[1])
        if kind == "helmet" and b[1] <= CLIP_PX:
            why.append("안전모 잘림 — 박스 안 침")
            continue
        picked.append((kind, s, b))
        if s < REVIEW_BELOW:
            why.append(f"{kind} 점수 낮음 {s:.2f}")
    labels[path.name] = picked
    notes[path.name] = why
    if i % 50 == 0:
        print(i, path.name, [(k, round(s, 2)) for k, s, _ in picked], why)

flagged = {n: w for n, w in notes.items() if w}
with (REVIEW / "review_list.csv").open("w", newline="", encoding="utf-8-sig") as f:
    wr = csv.writer(f)
    wr.writerow(["image", "helmet", "vest", "note"])
    for n in sorted(labels):
        kinds = {k: f"{s:.2f}" for k, s, _ in labels[n]}
        wr.writerow([n, kinds.get("helmet", ""), kinds.get("vest", ""), " / ".join(notes[n])])
both = sum(1 for v in labels.values() if {k for k, _, _ in v} == {"helmet", "vest"})
print(
    f"\n안전모+조끼 둘 다: {both}/400 · 검수 필요: {len(flagged)}장 → {REVIEW / 'review_list.csv'}"
)

In [ ]:
# 6. [A] 검수 시트 저장 + 라벨 확정 (애매한 사진 자동 제외)
import gc
import json

import cv2
import numpy as np

COLORS = {"helmet": (0, 140, 255), "vest": (60, 220, 60)}
TILE_W, TILE_H, COLS, ROWS = 270, 360, 5, 4


def tile(name):
    img = cv2.imread(str(IMG_DIR / name))
    for kind, s, b in labels[name]:
        x1, y1, x2, y2 = map(int, b)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[kind], 8)
        cv2.putText(
            img,
            f"{kind} {s:.2f}",
            (x1 + 6, max(40, y1 + 50)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.6,
            COLORS[kind],
            4,
        )
    h, w = img.shape[:2]
    r = min(TILE_W / w, (TILE_H - 30) / h)
    small = cv2.resize(img, (int(w * r), int(h * r)))
    canvas = np.full((TILE_H, TILE_W, 3), 255, np.uint8)
    canvas[30 : 30 + small.shape[0], : small.shape[1]] = small
    num = name[10:14]
    cv2.putText(
        canvas,
        num + (" CHECK" if notes[name] else ""),
        (4, 22),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255) if notes[name] else (0, 0, 0),
        2,
    )
    return canvas


names = sorted(labels)
per = COLS * ROWS
for k in range(0, len(names), per):
    tiles = [tile(n) for n in names[k : k + per]]
    tiles += [np.full((TILE_H, TILE_W, 3), 255, np.uint8)] * (per - len(tiles))
    sheet = np.vstack([np.hstack(tiles[r * COLS : (r + 1) * COLS]) for r in range(ROWS)])
    cv2.imwrite(
        str(REVIEW / f"sheet_{k // per + 1:02d}.jpg"), sheet, [cv2.IMWRITE_JPEG_QUALITY, 85]
    )
print("검수 시트", (len(names) + per - 1) // per, "장 →", REVIEW)

# 안전모·조끼 중 하나가 없거나 잘린 사진은 라벨을 비워 학습에서 뺀다 (반쪽 라벨은 «안 보이는 것» 으로 배운다).
# 점수만 낮은 사진은 빼지 않고 검수 시트에 CHECK 로만 표시한다 — 라벨러는 맞는 박스에도 0.3~0.5 를 자주 준다.
# 검수 시트를 본 뒤 틀린 사진이 있으면 번호를 EXCLUDE 에 적고 «런타임 → 모두 실행» 으로 다시 돌린다.
EXCLUDE_INCOMPLETE = True
EXCLUDE: list[int] = []  # 예: [12, 57, 203]


cats = [{"id": 0, "name": "ppe", "supercategory": "none"}] + [
    {"id": i + 1, "name": n, "supercategory": "ppe"}
    for i, n in enumerate(("helmet", "no_helmet", "vest", "no_vest"))
]
label_cat = {c["name"]: c["id"] for c in cats}
coco = {"images": [], "annotations": [], "categories": cats}
dropped = 0
for img_id, name in enumerate(names):
    w, h = Image.open(IMG_DIR / name).size
    coco["images"].append({"id": img_id, "file_name": name, "width": w, "height": h})
    if int(name[10:14]) in EXCLUDE or (
        EXCLUDE_INCOMPLETE and any("점수 낮음" not in w for w in notes[name])
    ):
        dropped += 1
        continue
    for kind, s, (x1, y1, x2, y2) in labels[name]:
        coco["annotations"].append(
            {
                "id": len(coco["annotations"]),
                "image_id": img_id,
                "category_id": label_cat[kind],
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "area": (x2 - x1) * (y2 - y1),
                "iscrowd": 0,
                "score": round(s, 3),
            }
        )
low_json.write_text(json.dumps(coco))
shutil.copy(low_json, REVIEW / "lowangle_annotations.coco.json")  # Roboflow 로 올려 고칠 수도 있다
print(f"라벨 사용 {len(names) - dropped}장 · 제외 {dropped}장 · 박스 {len(coco['annotations'])}개")

# 라벨러를 내려 GPU 메모리를 학습에 돌려준다
model = processor = None

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 7. [B] 런타임과 같은 사람 검출 (models/coco.onnx 와 같은 yolox_s.onnx · conf 0.5 · iou 0.45)
import cv2
import numpy as np
import onnxruntime as ort

STRIDES = (8, 16, 32)


def letterbox(img, size=640):
    """host/vision/detector.YoloxAdapter 와 같다 — BGR · 0~255 · 좌상단 정렬 · 여백 114 · 버림."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    canvas = np.full((size, size, 3), 114, np.uint8)
    canvas[: int(h * r), : int(w * r)] = cv2.resize(
        img, (int(w * r), int(h * r)), interpolation=cv2.INTER_LINEAR
    )
    return np.ascontiguousarray(canvas.transpose(2, 0, 1)[None], np.float32), r


def _grid(size=640):
    g, s = [], []
    for st in STRIDES:
        n = size // st
        xv, yv = np.meshgrid(np.arange(n), np.arange(n))
        g.append(np.stack((xv, yv), 2).reshape(-1, 2))
        s.append(np.full((n * n, 1), st))
    return np.concatenate(g), np.concatenate(s)


GRID, STRIDE = _grid()


def nms(boxes, scores, thr):
    order, keep = scores.argsort()[::-1], []
    areas = np.clip(boxes[:, 2] - boxes[:, 0], 0, None) * np.clip(
        boxes[:, 3] - boxes[:, 1], 0, None
    )
    while order.size:
        b = order[0]
        keep.append(b)
        rest = order[1:]
        iw = np.clip(
            np.minimum(boxes[b, 2], boxes[rest, 2]) - np.maximum(boxes[b, 0], boxes[rest, 0]),
            0,
            None,
        )
        ih = np.clip(
            np.minimum(boxes[b, 3], boxes[rest, 3]) - np.maximum(boxes[b, 1], boxes[rest, 1]),
            0,
            None,
        )
        inter = iw * ih
        union = areas[b] + areas[rest] - inter
        order = rest[np.where(union > 0, inter / np.maximum(union, 1e-9), 0) <= thr]
    return keep


def detect(sess, img, labels, conf=0.5, iou=0.45):
    """원시 [1, 8400, 5+C] → [(label, score, box)]. 클래스별 NMS (detector.Detector 와 같다)."""
    x, r = letterbox(img)
    raw = sess.run(None, {sess.get_inputs()[0].name: x})[0][0]
    c = (raw[:, :2] + GRID) * STRIDE
    half = np.exp(raw[:, 2:4]) * STRIDE / 2
    boxes = np.concatenate([c - half, c + half], 1) / r
    cls = raw[:, 5:].argmax(1)
    sc = raw[:, 4] * raw[np.arange(len(raw)), 5 + cls]
    m = sc >= conf
    boxes, sc, cls = boxes[m], sc[m], cls[m]
    h, w = img.shape[:2]
    boxes[:, 0::2] = boxes[:, 0::2].clip(0, w)
    boxes[:, 1::2] = boxes[:, 1::2].clip(0, h)
    out = []
    for k in np.unique(cls):
        mk = cls == k
        for i in nms(boxes[mk], sc[mk], iou):
            out.append((labels[k], float(sc[mk][i]), tuple(boxes[mk][i])))
    return sorted(out, key=lambda d: -d[1])


PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
coco_sess = ort.InferenceSession("/content/yolox_s.onnx", providers=PROVIDERS)
COCO_LABELS = ["person"] + [f"c{i}" for i in range(1, 80)]  # 0번만 쓴다

In [ ]:
# 8. [B] 하단각도 사진 → 사람 크롭 데이터 (기존 rf100_prepare 규칙)
import json
import random
import re

CLASSES = ("helmet", "no_helmet", "vest", "no_vest")
HEAD, TORSO = ("helmet", "no_helmet"), ("vest", "no_vest")
RUNTIME_PAD, TRAIN_PAD = 0.08, (0.0, 0.2)  # vision.ppe.crop_pad · 학습 분할 여유
MIN_PERSON_PX, MIN_KEEP_AREA, HEAD_MARGIN = 48, 0.5, 8
XIAO_LONG_SIDE = 640  # XIAO 는 VGA(640x480) — 폰 사진을 그 해상도로 낮춘다
VAL_BLOCKS = {2, 7}  # 400장을 촬영 순서로 40장씩 10구간 → 3·8번째 구간 = 검증
SEED = 20261001
rng = random.Random(SEED)


def norm(n):
    return re.sub(r"[\s\-]+", "_", n.strip().lower())


low = json.loads(low_json.read_text())
cat_name = {}
for c in low["categories"]:
    n = norm(c["name"])
    if n in CLASSES:
        cat_name[c["id"]] = n
    elif c.get("supercategory", "none") == "none" and not any(
        a["category_id"] == c["id"] for a in low["annotations"]
    ):
        continue  # Roboflow 상위 범주 자리표시자
    else:
        raise SystemExit(f"모르는 클래스 '{c['name']}' — 라벨 이름을 4개로 맞춘다")
anns_by_img = {}
for a in low["annotations"]:
    anns_by_img.setdefault(a["image_id"], []).append(a)


# area · inter 는 6번 셀에서 정의했다


def crop_rect(w, h, p, pad):
    """host/vision/ppe_detector.py 와 같은 식 — int 버림 · 클램프 · 8px 미만 실패."""
    dx, dy = (p[2] - p[0]) * pad, (p[3] - p[1]) * pad
    left, top = max(0, int(p[0] - dx)), max(0, int(p[1] - dy))
    right, bottom = min(w, int(p[2] + dx)), min(h, int(p[3] + dy))
    return None if right - left < 8 or bottom - top < 8 else (left, top, right, bottom)


old_cats = json.loads((ANN / "instances_train.json").read_text())["categories"]
cat_id = {c["name"]: c["id"] for c in old_cats}
assert set(cat_id) == set(CLASSES), f"기존 범주가 다르다: {cat_id}"

new = {"train": {"images": [], "annotations": []}, "val": {"images": [], "annotations": []}}
stats = {
    "사진": 0,
    "사람 미검출": 0,
    "사람 작음": 0,
    "머리·몸통 라벨 부족": 0,
    "머리 클리핑(검증 제외)": 0,
    "train": 0,
    "val": 0,
}
for im in low["images"]:
    stats["사진"] += 1
    m = re.search(r"PPE_lower_(\d{4})", im["file_name"])
    assert m, f"파일 이름에서 번호를 못 찾음: {im['file_name']}"
    idx = int(m.group(1))
    split = "val" if (idx - 1) // 40 in VAL_BLOCKS else "train"
    img = cv2.imread(str(low_json.parent / im["file_name"]))
    s = XIAO_LONG_SIDE / max(img.shape[:2])
    img = cv2.resize(
        img, (int(img.shape[1] * s), int(img.shape[0] * s)), interpolation=cv2.INTER_AREA
    )
    h, w = img.shape[:2]
    ppe = [
        (
            cat_name[a["category_id"]],
            tuple(
                v * s
                for v in (
                    a["bbox"][0],
                    a["bbox"][1],
                    a["bbox"][0] + a["bbox"][2],
                    a["bbox"][1] + a["bbox"][3],
                )
            ),
        )
        for a in anns_by_img.get(im["id"], [])
        if a["category_id"] in cat_name
    ]
    people = [d for d in detect(coco_sess, img, COCO_LABELS) if d[0] == "person"]
    if not people:
        stats["사람 미검출"] += 1
        continue
    person = max(people, key=lambda d: d[2][3] - d[2][1])[2]  # 런타임처럼 가장 큰 사람
    if person[3] - person[1] < MIN_PERSON_PX:
        stats["사람 작음"] += 1
        continue
    # ⚠️ 기존 규칙은 머리 0~30%·몸통 25~80% 구간으로 «그 사람의 라벨» 을 가린다. 하단각도는 몸통이
    # 위로 몰려 그 구간을 벗어나므로, 한 명만 찍힌 이 데이터에서는 사람 박스 안(IoA 0.5)만 본다.
    inside = [(c, b) for c, b in ppe if area(b) and area(inter(b, person)) / area(b) >= 0.5]
    if not any(c in HEAD for c, _ in inside) or not any(c in TORSO for c, _ in inside):
        stats["머리·몸통 라벨 부족"] += 1
        continue
    if split == "val" and person[1] <= HEAD_MARGIN:
        stats["머리 클리핑(검증 제외)"] += 1
        continue  # 런타임은 판정 보류라 검증에 넣지 않는다
    rect = crop_rect(w, h, person, rng.uniform(*TRAIN_PAD) if split == "train" else RUNTIME_PAD)
    if rect is None:
        continue
    crop = img[rect[1] : rect[3], rect[0] : rect[2]]
    quality = rng.randint(55, 90) if split == "train" else 75  # XIAO JPEG 압축 흉내
    name = f"low_{idx:04d}.jpg"
    folder = DATA / ("train2017" if split == "train" else "val2017")
    cv2.imwrite(str(folder / name), crop, [cv2.IMWRITE_JPEG_QUALITY, quality])
    ch, cw = crop.shape[:2]
    new[split]["images"].append({"file_name": name, "width": cw, "height": ch})
    for c, b in inside:
        cl = inter(b, rect)
        if area(cl) < area(b) * MIN_KEEP_AREA:
            continue
        x1, y1, x2, y2 = cl[0] - rect[0], cl[1] - rect[1], cl[2] - rect[0], cl[3] - rect[1]
        new[split]["annotations"].append(
            {
                "file_name": name,
                "category_id": cat_id[c],
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "area": (x2 - x1) * (y2 - y1),
                "iscrowd": 0,
            }
        )
    stats[split] += 1
print(stats)
cls_count = {}
for sp in new.values():
    for a in sp["annotations"]:
        cls_count[a["category_id"]] = cls_count.get(a["category_id"], 0) + 1
print("새 박스 수:", {c["name"]: cls_count.get(c["id"], 0) for c in old_cats})

In [ ]:
# 9. [B] 주석 합치기 — train_mix · val_mix(best 선택용) · val_low(하단각도만)
def merge(base_file, extra, out_file):
    base = (
        json.loads((ANN / base_file).read_text())
        if base_file
        else {"images": [], "annotations": [], "categories": old_cats}
    )
    img_id = max([i["id"] for i in base["images"]], default=0) + 1
    ann_id = max([a["id"] for a in base["annotations"]], default=0) + 1
    ids = {}
    for im in extra["images"]:
        base["images"].append({**im, "id": img_id})
        ids[im["file_name"]] = img_id
        img_id += 1
    for a in extra["annotations"]:
        a = {k: v for k, v in a.items() if k != "file_name"} | {
            "image_id": ids[a["file_name"]],
            "id": ann_id,
        }
        base["annotations"].append(a)
        ann_id += 1
    (ANN / out_file).write_text(json.dumps(base))
    print(out_file, len(base["images"]), "장")


merge("instances_train.json", new["train"], "instances_train_mix.json")
merge("instances_val.json", new["val"], "instances_val_mix.json")
merge(None, new["val"], "instances_val_low.json")

In [ ]:
# 10. [B] 학습 설정 — tools/ppe/yolox_exp_ppe_s.py 와 같고 fine-tune 값만 다르다
EXP = Path("/content/yolox_exp_ppe_s_ft.py")
EXP.write_text("""
import os, random
from pathlib import Path
import numpy as np
import yolox.layers
from pycocotools.cocoeval import COCOeval
from yolox.data import COCODataset
from yolox.data.data_augment import augment_hsv
from yolox.exp import Exp as BaseExp

yolox.layers.COCOeval_opt = COCOeval
HSV_GAINS = (3, 15, 30)   # 주황·형광 색 단서 보존 (MODEL_PLAN 1.3)

class WeakHsvCOCODataset(COCODataset):
    def pull_item(self, index):
        img, target, img_info, img_id = super().pull_item(index)
        img = np.ascontiguousarray(img).copy()
        augment_hsv(img, *HSV_GAINS)
        return img, target, img_info, img_id

class Exp(BaseExp):
    def __init__(self):
        super().__init__()
        self.depth, self.width, self.num_classes = 0.33, 0.50, 4
        self.exp_name = "yolox_exp_ppe_s_ft"
        self.output_dir = os.environ.get("PPE_OUT_DIR", "./YOLOX_outputs")  # Drive 에 바로 저장 — 연결이 끊겨도 남는다
        self.data_dir = os.environ["PPE_DATA_DIR"]
        self.train_ann = os.environ.get("PPE_TRAIN_ANN", "instances_train_mix.json")
        self.val_ann = os.environ.get("PPE_VAL_ANN", "instances_val_mix.json")
        self.test_ann = "instances_test.json"
        self.input_size = self.test_size = (640, 640)   # 바꾸면 출력이 [1, 8400, 9] 가 아니다
        self.multiscale_range = 2
        self.mosaic_prob, self.mosaic_scale = 0.5, (0.5, 1.5)
        self.enable_mixup, self.mixup_prob, self.hsv_prob = False, 0.0, 0.0
        self.flip_prob, self.degrees, self.translate, self.shear = 0.5, 5.0, 0.1, 1.0
        # ── fine-tune: 이미 수렴한 v4 에서 시작하므로 짧게·낮은 학습률로 ──
        self.max_epoch = 30
        self.no_aug_epochs = 10
        self.warmup_epochs = 1
        self.basic_lr_per_img = 0.01 / 64.0 * 0.3
        self.eval_interval = 5
        self.seed = 20261001

    def get_dataset(self, cache=False, cache_type="ram"):
        from yolox.data import TrainTransform
        return WeakHsvCOCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                                  preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=0.0),
                                  cache=cache, cache_type=cache_type)
""")
os.environ["PPE_DATA_DIR"] = str(DATA)
os.environ["PPE_OUT_DIR"] = str(OUT / "yolox_outputs")

In [ ]:
# 11. [B] 학습 (약 1시간) — 체크포인트는 Drive 에 매 epoch 저장
sh(
    f"cd /content/YOLOX && python tools/train.py -f {EXP} -d 1 -b 16 --fp16 -c '{CKPT_V4}' 2>&1 | tail -n 40"
)
BEST = (
    OUT / "yolox_outputs" / "yolox_exp_ppe_s_ft" / "best_ckpt.pth"
)  # 끊기면 latest_ckpt.pth 로 --resume

In [ ]:
# 12. [B] 평가 — v4 vs v5 · 기존 시험셋(v4 와 같은 잣대) · 하단각도 검증셋
def coco_eval(ckpt, val_ann):
    test = val_ann == "instances_test.json"  # --test 는 test2017 + exp.test_ann 을 쓴다
    env = {**os.environ, "PPE_VAL_ANN": val_ann}
    r = subprocess.run(
        f"cd /content/YOLOX && python tools/eval.py -f {EXP} -c '{ckpt}' -b 8 -d 1 --conf 0.001"
        + (" --test" if test else ""),
        shell=True,
        capture_output=True,
        text=True,
        env=env,
    )
    log = r.stdout + r.stderr
    m = re.search(r"Average Precision\s+\(AP\) @\[ IoU=0\.50\s+\|[^\]]*\]\s*=\s*([\d.]+)", log)
    return float(m.group(1)) if m else None, log


report = {}
for tag, ck in (("v4", CKPT_V4), ("v5", BEST)):
    for val in ("instances_test.json", "instances_val_low.json"):
        ap50, log = coco_eval(ck, val)
        report[f"{tag} · {val}"] = ap50
        (OUT / f"eval_{tag}_{val}.log").write_text(log)
for k, v in report.items():
    print(f"{k:40s} AP50 = {v}")

In [ ]:
# 13. [B] ONNX 내보내기 — export_ppe.py 와 같은 규칙 (원시 [1, 8400, 9] · opset 11)
import hashlib
import inspect
import sys

import torch
from torch import nn

sys.path.insert(0, "/content/YOLOX")
from yolox.exp import get_exp
from yolox.models.network_blocks import SiLU
from yolox.utils import replace_module


def export(ckpt, out):
    exp = get_exp(str(EXP), None)
    model = exp.get_model()
    model.load_state_dict(torch.load(str(ckpt), map_location="cpu", weights_only=False)["model"])
    model = replace_module(model.eval(), nn.SiLU, SiLU)
    model.head.decode_in_inference = False  # ⚠️ 디코드는 호스트 어댑터가 한다
    extra = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}
    torch.onnx.export(
        model,
        torch.zeros(1, 3, 640, 640),
        str(out),
        input_names=["images"],
        output_names=["output"],
        opset_version=11,
        **extra,
    )
    sess = ort.InferenceSession(str(out), providers=["CPUExecutionProvider"])
    x = np.random.default_rng(20260928).uniform(0, 255, (1, 3, 640, 640)).astype(np.float32)
    o = sess.run(None, {"images": x})[0]
    with torch.no_grad():
        diff = float(np.abs(o - model(torch.from_numpy(x)).numpy()).max())
    assert list(o.shape) == [1, 8400, 9], o.shape
    assert diff <= 1e-3, f"parity 초과 {diff}"
    sha = hashlib.sha256(out.read_bytes()).hexdigest()
    print(f"{out.name} · {out.stat().st_size:,} bytes · parity {diff:.1e} · sha256 {sha}")
    return sha


ONNX_V5 = OUT / "ppe_v5_lowangle.onnx"
ONNX_V4 = WORK / "ppe_v4_from_ckpt.onnx"
sha_v5 = export(BEST, ONNX_V5)
export(CKPT_V4, ONNX_V4)

In [ ]:
# 14. [B] 운용 판정 비교 — 하단각도 검증 크롭(전원 착용)에서 적합 / 위반(=오경보) / 판정 불가
def verdict(sess, crop):
    found = detect(sess, crop, list(CLASSES), conf=0.5)  # vision.ppe.conf_threshold
    heads = [d for d in found if d[0] in HEAD]
    torsos = [d for d in found if d[0] in TORSO]
    if not heads or not torsos:
        return "판정불가"
    return "위반(오경보)" if any(d[0] in ("no_helmet", "no_vest") for d in found) else "적합"


val_imgs = [DATA / "val2017" / im["file_name"] for im in new["val"]["images"]]
judge = {}
for tag, path in (("v4", ONNX_V4), ("v5", ONNX_V5)):
    sess = ort.InferenceSession(str(path), providers=PROVIDERS)
    counts = {"적합": 0, "위반(오경보)": 0, "판정불가": 0}
    for p in val_imgs:
        counts[verdict(sess, cv2.imread(str(p)))] += 1
    judge[tag] = counts
    print(tag, counts, f"(총 {len(val_imgs)}장)")

In [ ]:
# 15. [C] 깃허브 업로드 묶음 + 업로드 판정
from datetime import UTC, datetime

# Release 자산 이름은 v4 와 같게 둔다 (`ppe.onnx` · `ppe.json`) — fetch_models.py 는 태그·해시만 바꾸면 된다.
TAG = "ppe-v5"
GH = OUT / "github_upload"
REL = GH / f"release_{TAG}"
shutil.rmtree(GH, ignore_errors=True)
REL.mkdir(parents=True)
shutil.copy(ONNX_V5, REL / "ppe.onnx")
size_v5 = (REL / "ppe.onnx").stat().st_size

v4_meta = json.loads((HANDOVER / "ppe.json").read_text())
meta = {
    **{k: v4_meta[k] for k in ("family", "opset", "input", "output", "classes") if k in v4_meta},
    "name": "ppe",
    "version": "v5-lowangle",
    "sha256": sha_v5,
    "size_bytes": size_v5,
    "trained_at": datetime.now(UTC).astimezone().isoformat(timespec="seconds"),
    "base": {
        "version": v4_meta.get("version"),
        "onnx_sha256": v4_meta.get("sha256"),
        "checkpoint_sha256": hashlib.sha256(CKPT_V4.read_bytes()).hexdigest(),
    },
    "checkpoint": {"sha256": hashlib.sha256(BEST.read_bytes()).hexdigest()},
    "dataset": {
        "base": "ppe4_cs_md_v2 (v4 와 같음)",
        "added": {
            "name": "PPE_하단각도_학습이미지_400",
            "labeler": "Grounding DINO base (자동) · 검수 필요 사진 제외",
            "used_images": 400 - dropped,
            "excluded": dropped,
            "split": f"촬영 순서 40장 구간 {sorted(VAL_BLOCKS)} = 검증",
            "crop_stats": stats,
        },
        "train_ann": "instances_train_mix.json",
        "val_ann": "instances_val_mix.json",
        "test_ann": "instances_test.json",
    },
    "training": {
        "from": "ppe-v4 best_ckpt.pth",
        "max_epoch": 30,
        "no_aug_epochs": 10,
        "lr_scale": 0.3,
        "seed": 20261001,
        "hsv_gains": [3, 15, 30],
        "notebook": "tools/ppe/colab_ppe_v5_lowangle.ipynb",
    },
    "evaluation": {"coco_ap50": report, "lowangle_val_verdicts_conf0.5": judge},
    "status": "XIAO 실측 전 후보",
}
(REL / "ppe.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))
(REL / "NOTICE_addendum.txt").write_text(f"""PPE v5 (tag {TAG}) — v4 NOTICE 에 덧붙일 내용
- 베이스: ppe-v4 (YOLOX-S, Apache-2.0 Megvii) 에서 미세 조정
- 학습 데이터: ppe4_cs_md_v2 (Roboflow construction-safety · Mendeley PPE2286, 둘 다 CC BY 4.0)
  + 팀 자체 촬영 하단각도 사진 {400 - dropped}장 (사진은 배포하지 않는다)
- 자동 라벨러: Grounding DINO (IDEA-Research, Apache-2.0) — 라벨 생성에만 쓰고 배포하지 않는다
- 파일: ppe.onnx · {size_v5:,} bytes · sha256 {sha_v5}
""")

# 업로드 판정 — 기존 시험셋 AP50 이 0.01 넘게 떨어지거나, 하단각도 착용자 오경보가 늘거나 적합이 줄면 보류
ap_v4, ap_v5 = report.get("v4 · instances_test.json"), report.get("v5 · instances_test.json")
fa_v4, fa_v5 = judge["v4"]["위반(오경보)"], judge["v5"]["위반(오경보)"]
ok_v4, ok_v5 = judge["v4"]["적합"], judge["v5"]["적합"]
passed = (
    ap_v4 is not None
    and ap_v5 is not None
    and ap_v5 >= ap_v4 - 0.01
    and fa_v5 <= fa_v4
    and ok_v5 >= ok_v4
)
verdict_text = "✅ 업로드 권장" if passed else "⛔ 업로드 보류 — 아래 수치를 보고 팀장과 상의"

(GH / "REPO_CHANGES.md").write_text(f"""# PPE v5 반영 안내 — {verdict_text}

| | v4 | v5 |
| :--- | ---: | ---: |
| 기존 시험셋 AP50 (`instances_test.json`) | {ap_v4} | {ap_v5} |
| 하단각도 검증 AP50 | {report.get("v4 · instances_val_low.json")} | {report.get("v5 · instances_val_low.json")} |
| 하단각도 착용자 → 적합 | {ok_v4} | {ok_v5} |
| 하단각도 착용자 → 위반 (오경보) | {fa_v4} | {fa_v5} |
| 하단각도 착용자 → 판정 불가 | {judge["v4"]["판정불가"]} | {judge["v5"]["판정불가"]} |

⚠️ 공개 사진·폰 사진 기준이다. FR-9 판정은 XIAO 실측(`tools/ppe_live_check.py`)으로 한다.

## 1. GitHub Release — 새 태그 `{TAG}` (같은 태그 자산을 갈아끼우지 않는다)
`release_{TAG}/` 의 `ppe.onnx` · `ppe.json` 을 올리고, v4 의 `NOTICE.txt` 에 `NOTICE_addendum.txt` 를 덧붙여 올린다.

## 2. 저장소에서 고칠 곳
- `tools/fetch_models.py`
  - `PPE_RELEASE = "https://github.com/PJW-1/mechadog/releases/download/{TAG}/ppe.onnx"`
  - `models/ppe.onnx` 항목: `size={size_v5:_}` · `sha256="{sha_v5}"`
- `config/config.yaml` `vision.ppe` 학습 기록 주석: Release 태그 {TAG} · 하단각도 {400 - dropped}장 추가 · 크기 · sha256
- `models/NOTICE` §1 크기·SHA-256·학습일·태그, §3 에 자체 촬영 데이터 추가
- `models/README.md` 표의 `ppe-v4` → `{TAG}`

## 3. 커밋할 파일
- `tools/ppe/colab_ppe_v5_lowangle.ipynb` (이 노트북)

## 4. 올리지 않는 것
데이터셋 · `best_ckpt.pth` · 하단각도 사진 · `label_review/`
""")
print((GH / "REPO_CHANGES.md").read_text())
print("→ Drive", GH)